# FibroBlock — narrative walkthrough

**COE 562 Problem 8: cardiac action-potential propagation and conduction block.**

This notebook is a *thin wrapper*. Every calculation it shows is performed by the
library in `src/fibroblock/`; nothing is reimplemented here. That is deliberate:
if a notebook and a library can disagree, sooner or later they will, and the
figures in the report would stop matching the code that supposedly produced them.

The pipeline entry point is `scripts/make_all_figures.py`. **This notebook is not
part of it** — it exists to be read.

---

## Contents

1. [The model and its parameters](#1)
2. [Analytic results, computed at run time](#2)
3. [Numerical stability](#3)
4. [A single simulation](#4)
5. [Conduction velocity](#5)
6. [Conduction block](#6)
7. [Where to go next](#7)

In [1]:
import sys
from pathlib import Path

# Make the library importable whether or not it has been pip-installed.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from fibroblock import config as cfg
from fibroblock import fhn, grid, measure, operators, plotting, simulate, solvers, utils

print("fibroblock", utils.library_versions()["fibroblock"])
utils.set_seed(cfg.default_config().seed)

fibroblock 1.0.0
[seed] random seed fixed at 20260810 (model is deterministic)


Generator(PCG64) at 0x2ECF8406DC0

<a id="1"></a>
## 1. The model and its parameters

The monodomain cable equation with FitzHugh–Nagumo kinetics:

$$\frac{\partial V}{\partial t} = \frac{\partial}{\partial x}\!\left(D(x)\frac{\partial V}{\partial x}\right) + f(V,w),
\qquad \frac{\partial w}{\partial t} = \varepsilon(V + a - bw)$$

$$f(V,w) = V - \tfrac{1}{3}V^3 - w + I_{\text{stim}}(x,t)$$

Every parameter lives in `config.py` as a frozen dataclass. There are no magic
numbers anywhere else in `src/`.

In [2]:
config = cfg.default_config()

for section, values in config.to_dict().items():
    if isinstance(values, dict):
        print(f"{section}:")
        for key, value in values.items():
            print(f"    {key:<22} {value}")
    else:
        print(f"{section}: {values}")

fhn:
    a                      0.7
    b                      0.8
    eps                    0.08
    time_unit_ms           1.0
grid:
    length_cm              2.0
    dx_cm                  0.01
    baseline_D             0.001
gap:
    rho                    1.0
    gap_length_cm          0.1
    gap_centre_cm          1.0
    averaging              harmonic
stimulus:
    amplitude              1.0
    width_cm               0.1
    duration_ms            1.0
    start_ms               0.0
solver:
    dt_ms                  0.02
    t_end_ms               300.0
    method                 euler
    f_v_bound              3.0
    record_every           25
    store_full_history     False
measurement:
    activation_rule        v_zero_crossing
    activation_level       0.0
    cv_fit_skip_start_cm   0.5
    cv_fit_skip_end_cm     0.2
    block_margin_cm        0.3
    block_window_ms        200.0
seed: 20260810
label: default


**Units convention (assumption A1).** One dimensionless FitzHugh–Nagumo time
unit is *declared* equal to 1 ms. This is a calibration choice, not a
derivation — see `docs/assumption_register.md`.

<a id="2"></a>
## 2. Analytic results, computed at run time

None of these are hard-coded. Change `a` in the configuration and everything
below follows.

In [3]:
V_rest, w_rest = fhn.rest_state(config.fhn)
summary = fhn.excitability(config.fhn)
V1, V2, V3 = fhn.bistable_roots(config.fhn)

print(f"rest state       V* = {V_rest:.9f}   w* = {w_rest:.9f}")
print(f"Jacobian         tr = {summary.trace:+.6f}   det = {summary.determinant:+.6f}")
print(f"eigenvalues      {summary.eigenvalues[0]:.6f}, {summary.eigenvalues[1]:.6f}")
print(f"classification   {summary.classification}  ->  excitable = {summary.is_excitable}")
print()
print(f"bistable roots   V1 = {V1:.6f} (rest)")
print(f"                 V2 = {V2:.6f} (threshold)")
print(f"                 V3 = {V3:.6f} (excited)")
print(f"sum of roots     {V1 + V2 + V3:.2e}   (exactly zero: no V^2 term)")
print()
prefactor = fhn.analytic_cv_prefactor(config.fhn)
print(f"CV prefactor     theta/sqrt(D) = {prefactor:.6f}")
print(f"                 = -3*V2/sqrt(6) = {-3 * V2 / np.sqrt(6):.6f}   (same identity)")
theta_analytic = fhn.analytic_cv(config.grid.baseline_D, config.fhn)
print(f"analytic CV      {theta_analytic:.6f} cm/ms = {1000 * theta_analytic:.2f} cm/s")

rest state       V* = -1.199408035   w* = -0.624260044
Jacobian         tr = -0.502580   det = +0.108069
eigenvalues      -0.251290+0.211949j, -0.251290-0.211949j
classification   stable spiral  ->  excitable = True

bistable roots   V1 = -1.199408 (rest)
                 V2 = -0.786321 (threshold)
                 V3 = 1.985729 (excited)
sum of roots     0.00e+00   (exactly zero: no V^2 term)

CV prefactor     theta/sqrt(D) = 0.963043
                 = -3*V2/sqrt(6) = 0.963043   (same identity)
analytic CV      0.030454 cm/ms = 30.45 cm/s


**The identity worth remembering.** The frozen-$w$ cubic has no quadratic term,
so the roots sum to zero and $V_1 - 2V_2 + V_3 = -3V_2$. The front speed is
therefore controlled *entirely* by how far the excitation threshold $V_2$ sits
from rest. Push $V_2$ towards $V_1$ and the wave slows, then fails — that is the
mechanism of conduction block in one line.

<a id="3"></a>
## 3. Numerical stability

Von Neumann analysis gives $\Delta t \le 2/(4D_{\max}/\Delta x^2 + |f_V|_{\max})$,
binding at the checkerboard mode $k = \pi/\Delta x$.

In [4]:
limits = solvers.stability_limits(
    D_max=config.grid.baseline_D,
    dx=config.grid.dx_cm,
    f_v_bound=config.solver.f_v_bound,
    dt=config.solver.dt_ms,
)

print(f"4D/dx^2                  {limits.diffusion_number:.2f} per ms")
print(f"reaction-diffusion limit {limits.reaction_diffusion_dt_ms:.6f} ms")
print(f"pure-diffusion limit     {limits.pure_diffusion_dt_ms:.6f} ms")
print(f"  ... too optimistic by  {100 * limits.relative_overestimate:.2f} %")
print(f"working step             {limits.dt_ms} ms  (safety factor {limits.safety_factor:.2f})")
print(f"stable?                  {limits.is_stable}")

4D/dx^2                  40.00 per ms
reaction-diffusion limit 0.046512 ms
pure-diffusion limit     0.050000 ms
  ... too optimistic by  7.50 %
working step             0.02 ms  (safety factor 2.33)
stable?                  True


An unstable step is an **error**, not a warning. This is why no figure in the
report can have been produced by an unstable run by accident:

In [5]:
unstable = config.replace(solver=cfg.SolverParams(dt_ms=0.06, t_end_ms=10.0))
try:
    simulate.run_simulation(unstable)
except ValueError as error:
    print("Refused, as it should be:\n")
    print(error)

Refused, as it should be:

Time step dt = 0.06 ms exceeds the explicit-Euler stability limit of 0.046512 ms (D_max = 0.001 cm^2/ms, dx = 0.01 cm, |f_V|_max = 3.0). Reduce dt, or pass force=True if the instability is the point.


<a id="4"></a>
## 4. A single simulation

One healthy strand, 300 ms.

In [6]:
healthy = config.replace(gap=cfg.GapParams(rho=1.0, gap_length_cm=0.0))
result = simulate.run_simulation(healthy)

print(f"{result.n_steps_taken} steps in {result.wall_seconds:.2f} s")
print(f"diverged: {result.diverged}")
print(f"nodes activated: {np.isfinite(result.activation_time_crossing).sum()} / {result.grid.n_nodes}")
print(f"peak V: {result.V_peak.max():.4f}")

15000 steps in 0.48 s
diverged: False
nodes activated: 201 / 201
peak V: 1.7952


In [7]:
fig, (left, right) = plotting.new_figure(figsize=(11, 4), ncols=2, constrained_layout=True)

for target in (10.0, 30.0, 50.0, 70.0):
    index = int(np.argmin(np.abs(result.snapshot_times - target)))
    left.plot(result.x, result.V_snapshots[index], label=f"$t={result.snapshot_times[index]:.0f}$ ms")
plotting.label_axes(left, "position $x$ (cm)", "$V$ (dimensionless)", "Propagating action potential")
left.legend(fontsize=8)

mesh = plotting.space_time_image(right, result.x, result.snapshot_times, result.V_snapshots)
right.set_title("Space-time map")
fig.colorbar(mesh, ax=right, label="$V$ (dimensionless)")
plt.show()

C:\Users\dippe\AppData\Local\Temp\ipykernel_16260\2778346186.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<a id="5"></a>
## 5. Conduction velocity

Fitted over the steady-propagation window only: the first 0.5 cm is discarded
(the wave is still forming) and the last 0.2 cm too (the sealed end reflects).

In [8]:
fit = measure.measure_velocity(result)

print(f"measured   {fit.theta_cm_per_ms:.6f} cm/ms = {fit.theta_cm_per_s:.2f} cm/s")
print(f"analytic   {theta_analytic:.6f} cm/ms = {1000 * theta_analytic:.2f} cm/s")
print(f"difference {100 * (fit.theta_cm_per_ms - theta_analytic) / theta_analytic:+.2f} %")
print(f"R^2        {fit.r_squared:.8f}  over {fit.n_points} nodes")
print(f"window     [{fit.window_start_cm}, {fit.window_end_cm}] cm, rule = {fit.rule}")

measured   0.025534 cm/ms = 25.53 cm/s
analytic   0.030454 cm/ms = 30.45 cm/s
difference -16.16 %
R^2        1.00000000  over 131 nodes
window     [0.5, 1.8] cm, rule = v_zero_crossing


The measured velocity sits **about 16 % below** the analytic value. This is
physical, not numerical — `ex03` shows the default grid carries only 0.52 %
discretisation error. The analytic derivation freezes $w$ at its resting value,
but $w$ has already risen by the time the front arrives:

In [9]:
w_front = measure.recovery_at_front(result, probe_x_cm=1.2, threshold_V=V2)
corrected = fhn.front_speed_prefactor_at(w_front)

print(f"w at rest            {w_rest:.6f}  ->  prefactor {prefactor:.6f}")
print(f"w measured at front  {w_front:.6f}  ->  prefactor {corrected:.6f}")
print(f"measured prefactor   {fit.theta_cm_per_ms / np.sqrt(config.grid.baseline_D):.6f}")

w at rest            -0.624260  ->  prefactor 0.963043
w measured at front  -0.590765  ->  prefactor 0.869739
measured prefactor   0.807444


<a id="6"></a>
## 6. Conduction block

Now insert a 0.1 cm patch of reduced coupling at the centre of the strand and
vary $\rho$. The block criterion is stated in the result object itself, so any
saved result carries its own definition.

In [10]:
for rho in (1.0, 0.5, 0.2, 0.16, 0.15, 0.05):
    gapped = config.replace(
        gap=cfg.GapParams(rho=rho, gap_length_cm=0.1, gap_centre_cm=1.0),
        solver=cfg.SolverParams(dt_ms=0.02, t_end_ms=205.0, record_every=50),
    )
    run = simulate.run_simulation(gapped)
    verdict = measure.detect_block(run)
    delay = measure.measure_delay(run)
    transit = f"{delay.transit_ms:6.2f} ms" if delay.propagated else "      -   "
    print(f"rho = {rho:<5}  blocked = {str(verdict.blocked):<5}  transit = {transit}"
          f"  furthest activation = {verdict.furthest_activation_cm:.2f} cm")

print()
print(verdict.criterion)

rho = 1.0    blocked = False  transit =  27.41 ms  furthest activation = 2.00 cm
rho = 0.5    blocked = False  transit =  29.50 ms  furthest activation = 2.00 cm


rho = 0.2    blocked = False  transit =  35.14 ms  furthest activation = 2.00 cm


rho = 0.16   blocked = False  transit =  38.83 ms  furthest activation = 2.00 cm


rho = 0.15   blocked = True   transit =       -     furthest activation = 1.09 cm


rho = 0.05   blocked = True   transit =       -     furthest activation = 1.05 cm

blocked if no node at x > 1.3500 cm reaches V = 0.0 within 200.0 ms of the stimulus


The threshold sits between $\rho = 0.15$ and $0.16$ for this gap length. `ex06`
locates it by bisection to $\rho_{\text{crit}} = 0.1573$, and maps the whole
curve in the $(L_{\text{gap}}, \rho)$ plane.

**The threshold is a curve, not a number.** It rises from 0.067 at a single-node
gap to 0.157 and then saturates.

### The averaging scheme is not a free choice

Interface conductances must use the **harmonic** mean, because $D \propto 1/r$
and series resistances add. Here is what the arithmetic mean does as coupling
falls:

In [11]:
D0 = config.grid.baseline_D
print(f"{'rho':>8} {'harmonic':>14} {'arithmetic':>14}")
for rho in (0.5, 0.1, 0.01, 0.001, 0.0001):
    left = np.array([D0])
    right = np.array([rho * D0])
    print(f"{rho:>8} {grid.harmonic_mean(left, right)[0] / D0:>14.6f}"
          f" {grid.arithmetic_mean(left, right)[0] / D0:>14.6f}")
print()
print("As rho -> 0 the harmonic mean tends to 0, as a series resistance must.")
print("The arithmetic mean tends to D0/2 -- half the healthy conductance survives,")
print("no matter how completely the tissue is uncoupled. A single-node gap then")
print("cannot be blocked at ANY coupling ratio (see ex06).")

     rho       harmonic     arithmetic
     0.5       0.666667       0.750000
     0.1       0.181818       0.550000
    0.01       0.019802       0.505000
   0.001       0.001998       0.500500
  0.0001       0.000200       0.500050

As rho -> 0 the harmonic mean tends to 0, as a series resistance must.
The arithmetic mean tends to D0/2 -- half the healthy conductance survives,
no matter how completely the tissue is uncoupled. A single-node gap then
cannot be blocked at ANY coupling ratio (see ex06).


<a id="7"></a>
## 7. Where to go next

| Want | Look at |
|---|---|
| All eight figures | `python scripts/make_all_figures.py` (≈6 min) |
| Verification evidence | `docs/verification_log.md` (47 checks) |
| Why each numerical choice | `docs/numerical_choices.md` |
| What is assumed | `docs/assumption_register.md` (17 assumptions) |
| Module-by-module tour | `docs/code_walkthrough.md` |
| Comparison with literature | `docs/validation_log.md` |

Run the test suite with `pytest` — 93 tests, no skips. Each test's docstring
states which report claim it supports.